# **Network Dissection своими руками**

Практика к уроку про Network Dissection. Мы не запускаем авторский код — мы собираем саму меру
из трёх решений, которые разобраны в уроке: квантильный порог активации, растягивание маски до
размера входа и IoU, посчитанный объединением по всему набору, а не усреднением по картинкам.

## Сколько это стоит

**Полный Network Dissection** ([NetDissect-Lite](https://github.com/CSAILVision/NetDissect-Lite)):
датасет Broden — около **1 ГБ** на диске; разбор одной сети на видеокарте — **порядка 20 минут**
для ResNet18 и **около двух часов** для DenseNet161. Код писался под Python 3.6 и проверялся на
Ubuntu 16.04, у репозитория два десятка открытых вопросов и давно нет обновлений: на нынешних
версиях PyTorch запуск требует правки чужого кода. Метод не забагован — устарел его стек.

**Эта тетрадь:** CIFAR-10 — около **170 МБ**, предобученная ResNet18 — **45 МБ**, весь расчёт на
200 изображениях — **две-три минуты на CPU**, видеокарта не нужна. Мы платим тем, что концепции
у нас не размечены человеком, а заданы правилом по цвету. Это огрубление, и оно честно названо:
настоящий Broden размечен по пикселям людьми и содержит около полутора тысяч понятий.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("считаем на", device)

## Шаг 1. Сеть и набор изображений

Берём ResNet18, обученную на ImageNet, и смотрим на последний свёрточный блок `layer4`: там
512 каналов и карта $7\times7$ — то самое разрешение, о котором урок говорит «разглядывать
нечего». Изображения берём из CIFAR-10 и растягиваем до 224 пикселей, как ждёт сеть.

In [ ]:
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights).eval().to(device)

tf = transforms.Compose([transforms.Resize(224), transforms.ToTensor()])
ds = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=tf)
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(ds, range(200)), batch_size=25)

norm = transforms.Normalize(mean=weights.transforms().mean, std=weights.transforms().std)

acts, imgs = [], []
hook = model.layer4.register_forward_hook(lambda m, i, o: acts.append(o.detach().cpu()))
with torch.no_grad():
    for x, _ in loader:
        imgs.append(x)
        model(norm(x).to(device))
hook.remove()

A = torch.cat(acts)      # активации: (N, K, 7, 7)
X = torch.cat(imgs)      # картинки:  (N, 3, 224, 224)
print("активации", tuple(A.shape), "| изображения", tuple(X.shape))

## Шаг 2. Концепция вместо разметки Broden

У нас нет попиксельной разметки, поэтому концепцию задаём правилом: пиксель относится к понятию
«зелень», если зелёный канал заметно превосходит красный и синий. Это и есть $L_c(x_i)$ из
формулы урока — множество пикселей, «аннотированных» для понятия $c$.

Замена не безобидная, и в этом весь смысл ограничения, о котором говорит урок: **мы измеряем не
то, что выучила сеть, а пересечение выученного с нашим списком понятий.** Список из одного
цветового правила — крайний случай такого списка.

In [ ]:
concept = (X[:, 1] > X[:, 0] + 0.06) & (X[:, 1] > X[:, 2] + 0.06)   # L_c(x_i)
print("доля пикселей концепции по набору:", round(concept.float().mean().item(), 4))

## Шаг 3. Три решения, спрятанные в формуле

Собираем меру ровно так, как разобрано в уроке.

1. **Порог — квантиль, а не абсолютное число.** $T_k$ выбирается так, что $P(a_k > T_k) = 0.005$,
   и он свой для каждого канала: активации разных каналов живут в разных диапазонах.
2. **Растягиваем маску, а не сжимаем разметку.** Карта $7\times7$ поднимается до $224\times224$
   билинейной интерполяцией. Обратный путь потерял бы всё, что мельче области $32\times32$.
3. **Объединение по набору, а не среднее по картинкам.** Числитель и знаменатель копятся по всем
   изображениям, и только потом берётся отношение — иначе канал, идеально сработавший на трёх
   картинках из двухсот, получил бы высокий средний IoU.

In [ ]:
def iou_for_channel(k, quantile=0.005):
    """IoU канала k с концепцией — по формуле из урока."""
    a = A[:, k]
    T = torch.quantile(a.flatten().float(), 1 - quantile)                 # (1) квантильный порог

    mask = F.interpolate(a.unsqueeze(1), size=(224, 224),                 # (2) растягиваем маску
                         mode="bilinear", align_corners=False)[:, 0] > T

    inter = (mask & concept).sum().item()                                 # (3) объединяем по набору
    union = (mask | concept).sum().item()
    return inter / union if union else 0.0


scores = np.array([iou_for_channel(k) for k in range(A.shape[1])])
order = scores.argsort()[::-1]

print("каналов всего:", len(scores))
print("медиана IoU по каналам:", round(float(np.median(scores)), 5))
for k in order[:5]:
    print(f"  канал {k:>3}: IoU = {scores[k]:.4f}")

Посмотрите на разрыв: у верхних каналов IoU в десятки раз выше медианы. Это и есть
утверждение метода — отклик отдельных единиц действительно совпадает с понятием, а не размазан
по всей сети равномерно.

А теперь на абсолютные числа. Порог детектора в статье — **0.04**. Наш лучший канал до него
скорее всего не дотянул. Причина названа в уроке: билинейная интерполяция размывает края маски,
и посчитанный IoU выходит ниже настоящего — особенно для мелких объектов.

In [ ]:
best = int(order[0])
print(f"лучший канал: {best}, IoU = {scores[best]:.4f}, порог детектора в статье 0.04")
print("детектором объявлен бы:", scores[best] >= 0.04)

**Задание 1.** Поднимите долю активных пикселей с 0.005 до 0.02 (то есть опустите порог
$T_k$) и пересчитайте. Что происходит с IoU лучших каналов и почему? Урок предсказывает ответ —
проверьте предсказание числом.

In [ ]:
# Ваш код здесь

`Ваш ответ здесь.`

**Задание 2.** Замените концепцию «зелень» на «синева» (синий канал заметно больше красного
и зелёного). Совпадают ли верхние каналы с теми, что нашлись для зелени? Что означал бы ответ
«совпадают» — для метода и для утверждения «одна единица — одна концепция»?

In [ ]:
# Ваш код здесь

`Ваш ответ здесь.`

## Шаг 4. Посмотреть глазами на то, что посчитано

Числу верить рано, пока не увидели маску. Ниже — изображения, где лучший канал сработал сильнее
всего, и его маска поверх картинки.

In [ ]:
k = best
a = A[:, k]
T = torch.quantile(a.flatten().float(), 1 - 0.005)
mask = F.interpolate(a.unsqueeze(1), size=(224, 224), mode="bilinear", align_corners=False)[:, 0] > T

top_imgs = a.amax(dim=(1, 2)).argsort(descending=True)[:5]
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for col, idx in enumerate(top_imgs):
    axes[0, col].imshow(X[idx].permute(1, 2, 0).numpy())
    axes[0, col].set_title(f"top-{col + 1}")
    axes[1, col].imshow(X[idx].permute(1, 2, 0).numpy())
    axes[1, col].imshow(mask[idx].numpy(), alpha=0.45, cmap="autumn")
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle(f"канал {k}: сверху изображение, снизу маска активации")
plt.tight_layout();

**Задание 3.** Совпадает ли маска с зелёными областями на глаз? Найдите изображение, где
канал сработал сильно, а зелени в нём мало, и объясните, что это говорит о подписи «канал $k$ —
детектор зелени».

In [ ]:
# Ваш код здесь

`Ваш ответ здесь.`

## Что мы сделали и чего не сделали

Мы собрали меру Network Dissection целиком и получили осмысленный порядок каналов за пару минут
вместо гигабайта разметки и двух часов расчёта. Чего у нас нет: настоящего словаря понятий.
Полторы тысячи размеченных людьми концепций Broden — это и сила метода, и его потолок, о который
он упирается: чего нет в разметке, того метод не увидит.

Способ обойти потолок разбирается в уроке про CLIP-Dissect, и практика к нему — отдельная
тетрадь `PRACTICE_clip_dissect`. Там словарь задаётся текстом, а масок нет вовсе.